# Phase 9C — quantitative MPRA activity, frozen locus holdout

**Kaggle setup:** Select a GPU accelerator (T4/P100 or better) and turn Internet ON. Attach the completed Phase 9A output, **`9_enhancer_phase9a_full_activity_dataset.zip`**, as the only required Input. A Kaggle-unpacked version of that output is accepted. Phase 9B weights and results are neither inputs nor valid starting checkpoints for this split.

This notebook runs the companion **Phase 9A locus-group split** on the full continuous-activity data. For each cell line it fits GC/CpG ridge, strand-invariant spaced-6/4-mer ridge, a one-hot CNN with three prespecified seeds, and pretrained DNABERT-2 regression with three prespecified seeds. It uses train-only target scaling, validation-only choice of ridge alpha or neural epoch, reverse-complement augmentation and inference, record-level predictions, held-out metrics, a locus-group paired bootstrap, and saved model checkpoints. The model settings and seeds are the same as Phase 9B; this run trains fresh models using the Phase 9A train/validation/test partition.

**Run all cells in order.** Neural training can take a long time. Saving a Kaggle version preserves completed jobs under `enhancer_phase9c_work`; add that *Phase 9C* notebook Output as an optional Input to resume an interrupted run. Download both resulting files: **`enhancer_phase9c_regression_results.zip`** (tables and provenance) and **`enhancer_phase9c_complete_with_weights.zip`** (the results and all 12 checkpoints). The complete bundle may be several GB. Keep both and the saved Kaggle version until the files have been downloaded and checked.

**Interpretation:** Phase 9B's chromosome test and Phase 9C's locus test contain different records; compare their metrics as separate evaluations, not as paired predictions. HepG2 and K562 represent different ENCODE assay/design families, and the target is mean log2 MPRA activity, not binary enhancer status. Exact and reverse-complement grouping is verified below, but it does not establish absence of every near-duplicate or distant homolog.


In [1]:
# 0 — Imports, frozen protocol and Input discovery.
import os,sys,io,json,zipfile,hashlib,gc,random,shutil,subprocess,warnings,math
from pathlib import Path
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
import numpy as np,pandas as pd
from scipy import sparse
from scipy.stats import pearsonr,spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler,Normalizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics import r2_score,mean_squared_error,mean_absolute_error
from itertools import combinations
ROOT=Path('/kaggle/input');assert ROOT.exists(),'Attach the completed Phase 9A ZIP as an Input.'
WORK=Path('/kaggle/working/enhancer_phase9c_work');WORK.mkdir(parents=True,exist_ok=True)
CACHE=WORK/'cache';CACHE.mkdir(exist_ok=True)
WEIGHTS=WORK/'checkpoints';WEIGHTS.mkdir(exist_ok=True)
PRED=WORK/'predictions';PRED.mkdir(exist_ok=True)
EXPECTED={
 'phase9a_audit_and_provenance.json':'39118ed7b2dc50631f87dcb0c470249c649609d3dd7ab8c885b10064c66deff2',
 'phase9_full_activity_all_cells.csv':'4a359a715f7a3f1fbcf0d044cc08446137df584c9ab4a43a2de18637bb341a76',
 'phase9_frozen_splits.csv':'1ecc2d271817edd3000890be718a3f3a9bcc6b63a71550dfa718dbed84d0ff61'}
PHASE9A_PROTOCOL='5e438773a3f9c74cf006d183bcfc30daabd3d85bdbfefd47acfa48dc023e450a'
REVISION='ec1f874253852eb3907081f57294991b4280ceb6'
CONFIG={'name':'phase9c_locus_regression_v1','phase9a_protocol':PHASE9A_PROTOCOL,
 'phase9a_data_sha256':EXPECTED['phase9_full_activity_all_cells.csv'],
 'phase9a_split_sha256':EXPECTED['phase9_frozen_splits.csv'],'split':'split_9a',
 'classical_alphas':{'gc_cpg':[0.1,1.,10.], 'spaced6_4':[1.,10.,100.]},
 'seeds':[20260930,20260931,20260932],
 'cnn_epochs':6,'cnn_batch':128,'cnn_lr':0.001,
 'dnabert2_epochs':2,'dnabert2_batch':8,'dnabert2_accumulation':2,
 'dnabert2_lr':2e-5,'dnabert2_revision':REVISION,'dnabert2_max_tokens':256,
 'selection':'highest validation Pearson; tie lowest alpha or earliest epoch',
 'rc_policy':'strand-invariant classical; randomly oriented neural training per epoch; mean forward and RC inference',
 'bootstrap_repeats':300,'bootstrap_seed':20261013}
def sha(b):return hashlib.sha256(b).hexdigest()
def fsha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
 return h.hexdigest()
LOCK=sha(json.dumps(CONFIG,sort_keys=True,separators=(',',':')).encode())
print('Phase 9C protocol:',LOCK)

def locked_member(name):
 found=[]
 for p in ROOT.rglob(name):
  if p.is_file():found.append((p.read_bytes(),str(p)))
 for p in ROOT.rglob('*.zip'):
  try:
   with zipfile.ZipFile(p) as z:
    matches=[m for m in z.namelist() if m==name or m.endswith('/'+name)]
    if len(matches)==1:found.append((z.read(matches[0]),str(p)+'!'+matches[0]))
  except (zipfile.BadZipFile,OSError):pass
 good=[(b,where) for b,where in found if sha(b)==EXPECTED[name]]
 if not good:raise RuntimeError(f'Missing or unexpected {name}; attach the exact completed Phase 9A Output. Seen: '+str([(w,sha(b)) for b,w in found]))
 print('Verified',name,'from',good[0][1]);return good[0][0]
raw_audit=locked_member('phase9a_audit_and_provenance.json')
raw_data=locked_member('phase9_full_activity_all_cells.csv')
raw_split=locked_member('phase9_frozen_splits.csv')
audit=json.loads(raw_audit)
assert audit['protocol_hash']==PHASE9A_PROTOCOL
assert audit['output_sha256']['phase9_full_activity_all_cells.csv']==sha(raw_data)
assert audit['output_sha256']['phase9_frozen_splits.csv']==sha(raw_split)
d=pd.read_csv(io.BytesIO(raw_data),dtype={'record_id':str,'sequence':str,'sequence_sha256':str,'locus_group_id':str,'split_group_id':str})
locked=pd.read_csv(io.BytesIO(raw_split),dtype=str,keep_default_na=False)
assert len(d)==76833 and d.record_id.is_unique and len(locked)==len(d) and locked.record_id.is_unique
assert d.record_id.tolist()==locked.record_id.tolist()
for col in ('cell_line','locus_group_id','split_group_id','split_9a','split_9a'):
 assert d[col].astype(str).tolist()==locked[col].tolist(),col
# The 9B chromosome quarantine flag is specific to the other partition.
assert d.sequence.str.fullmatch('[ACGT]{200}').all()
assert all(sha(s.encode())==h for s,h in zip(d.sequence,d.sequence_sha256))
assert np.isfinite(d.mean_log2_activity).all()
assert d.groupby('locus_group_id').split_9a.nunique().max()==1
assert d.groupby('split_group_id').split_9a.nunique().max()==1
assert d.groupby('sequence_sha256').split_9a.nunique().max()==1
# The canonical identifier joins each sequence with its reverse complement.
assert all(sha(min(s,s.translate(str.maketrans('ACGT','TGCA'))[::-1]).encode())==h
 for s,h in zip(d.sequence,locked.sequence_canonical_sha256))
assert locked.groupby('sequence_canonical_sha256').split_9a.nunique().max()==1
assert d.groupby('split_group_id').split_9a.nunique().max()==1
assert set(d.split_9a)=={'train','validation','test'}
expected_counts={('HepG2','train'):21307,('HepG2','validation'):5292,('HepG2','test'):6725,
 ('K562','train'):27931,('K562','validation'):6917,('K562','test'):8661}
assert d.groupby(['cell_line','split_9a']).size().to_dict()==expected_counts
assert d.groupby(['cell_line','split_9a']).size().min()>100
print(d.groupby(['cell_line','split_9a']).size())
# Import finished outputs from a prior Save Version, if supplied as an additional Input.
for folder in (CACHE,WEIGHTS,PRED):
 prior=[p for p in ROOT.rglob('*') if p.is_file() and p.parent.name==folder.name and
        p.parent.parent.name=='enhancer_phase9c_work']
 for name in sorted(set(p.name for p in prior)):
  choices=[p for p in prior if p.name==name]
  assert len({fsha(p) for p in choices})==1,f'Ambiguous prior artifact: {name}'
  target=folder/name
  if target.exists():assert fsha(target)==fsha(choices[0]),f'Conflicting previous {name}'
  else:shutil.copy2(choices[0],target)
print('Previous completed artifacts imported:',sum(len(list(p.iterdir())) for p in (CACHE,WEIGHTS,PRED)))


Phase 9C protocol: dd2b4c43281df1782900f5b11265b80d1539c635ee261fe82d07f70a8705f610
Verified phase9a_audit_and_provenance.json from /kaggle/input/datasets/aryanaditta/9-enhancer-phase9a-full-activity-dataset-zip/phase9a_audit_and_provenance.json
Verified phase9_full_activity_all_cells.csv from /kaggle/input/datasets/aryanaditta/9-enhancer-phase9a-full-activity-dataset-zip/phase9_full_activity_all_cells.csv
Verified phase9_frozen_splits.csv from /kaggle/input/datasets/aryanaditta/9-enhancer-phase9a-full-activity-dataset-zip/phase9_frozen_splits.csv
cell_line  split_9a  
HepG2      test           6725
           train         21307
           validation     5292
K562       test           8661
           train         27931
           validation     6917
dtype: int64
Previous completed artifacts imported: 80


In [2]:
# 1 — Shared helpers: patient split access, metrics, per-job caches.
BASE_ARMS=['gc_cpg_ridge','spaced6_4_ridge']
SEEDS=CONFIG['seeds']
NEURAL_ARMS=[f'{family}_seed_{seed}' for family in ('cnn','dnabert2') for seed in SEEDS]
ENSEMBLES=['cnn_three_seed_mean','dnabert2_three_seed_mean']
ALL_ARMS=BASE_ARMS+NEURAL_ARMS+ENSEMBLES
COMP=str.maketrans('ACGT','TGCA')
def rc(s):return s.translate(COMP)[::-1]
def partitions(cell):
 sub=d[d.cell_line.eq(cell)]
 return tuple(sub[sub.split_9a.eq(split)].reset_index(drop=True) for split in ('train','validation','test'))
def correlation(y,p):
 if np.std(y)==0 or np.std(p)==0:return -1.0
 return float(np.corrcoef(np.asarray(y,dtype=float),np.asarray(p,dtype=float))[0,1])
def measure(y,p):
 y=np.asarray(y,dtype=float);p=np.asarray(p,dtype=float)
 assert len(y)>3 and len(y)==len(p) and np.isfinite(p).all()
 return {'n':int(len(y)),'pearson':correlation(y,p),
  'spearman':float(spearmanr(y,p).statistic),'r2':float(r2_score(y,p)),
  'rmse':float(np.sqrt(mean_squared_error(y,p))),'mae':float(mean_absolute_error(y,p))}
def write_atomic_csv(frame,path):
 tmp=path.with_suffix('.part');frame.to_csv(tmp,index=False,float_format='%.12g');os.replace(tmp,path)
def write_atomic_json(obj,path):
 tmp=path.with_suffix('.part');tmp.write_text(json.dumps(obj,indent=2,sort_keys=True)+'\n');os.replace(tmp,path)
def save_validation(cell,arm,va,pred,choice,checkpoint=None):
 pred=np.asarray(pred,dtype=float)
 assert len(pred)==len(va) and np.isfinite(pred).all()
 p=PRED/f'{cell}_{arm}_validation.csv';meta=CACHE/f'{cell}_{arm}_validation_meta.json'
 write_atomic_csv(pd.DataFrame({'record_id':va.record_id,'prediction':pred}),p)
 entry={'lock':LOCK,'cell_line':cell,'model':arm,'validation_sha256':fsha(p),
        'choice':choice,'checkpoint_sha256':fsha(checkpoint) if checkpoint else None,
        'checkpoint_file':checkpoint.name if checkpoint else None}
 write_atomic_json(entry,meta)
def load_validation(cell,arm,va,checkpoint=None):
 p=PRED/f'{cell}_{arm}_validation.csv';meta=CACHE/f'{cell}_{arm}_validation_meta.json'
 if not(p.exists() and meta.exists()):return None
 m=json.loads(meta.read_text())
 assert m['lock']==LOCK and m['cell_line']==cell and m['model']==arm
 assert fsha(p)==m['validation_sha256']
 if checkpoint:
  assert checkpoint.exists() and m['checkpoint_file']==checkpoint.name
  assert fsha(checkpoint)==m['checkpoint_sha256']
 pred=pd.read_csv(p,dtype={'record_id':str,'prediction':float})
 assert pred.record_id.tolist()==va.record_id.tolist() and np.isfinite(pred.prediction).all()
 return pred.prediction.to_numpy(),m['choice']


In [3]:
# 2 — CPU baselines, validation selection only. Spaced 6/4-mers average both strands.
MASKS=np.asarray(list(combinations(range(6),4)),dtype=np.int16)
WEIGHT=np.array([64,16,4,1],dtype=np.int32)
BASE_MAP=np.zeros(256,dtype=np.uint8)
for base,num in zip(b'ACGT',range(4)):BASE_MAP[base]=num

def feature_gc(seqs):
 return np.asarray([[(s.count('G')+s.count('C'))/200.,s.count('CG')/199.] for s in seqs],dtype=np.float32)
def feature_spaced(seqs):
 # Fixed, interpretable 15-mask x 256-word vocabulary; no labels used here.
 rows=[]
 for start in range(0,len(seqs),256):
  batch=np.zeros((min(256,len(seqs)-start),3840),dtype=np.float32)
  for j,seq in enumerate(seqs[start:start+256]):
   values=BASE_MAP[np.frombuffer(seq.encode(),dtype=np.uint8)]
   for strand in (values,3-values[::-1]):
    windows=np.lib.stride_tricks.sliding_window_view(strand,6)
    words=(windows[:,MASKS].astype(np.int32)*WEIGHT).sum(-1)
    words+=256*np.arange(15,dtype=np.int32)[None,:]
    batch[j]+=np.bincount(words.ravel(),minlength=3840).astype(np.float32)
  rows.append(sparse.csr_matrix(batch))
 return sparse.vstack(rows,format='csr')
def classical(cell,arm):
 tr,va,te=partitions(cell)
 cached=load_validation(cell,arm,va)
 if cached:
  print('REUSED baseline selection',cell,arm);return cached[1]
 y=tr.mean_log2_activity.to_numpy(dtype=float)
 if arm=='gc_cpg_ridge':
  x=feature_gc(tr.sequence);v=feature_gc(va.sequence)
  normalizer=StandardScaler().fit(x);x=normalizer.transform(x);v=normalizer.transform(v)
  grid=CONFIG['classical_alphas']['gc_cpg']
 elif arm=='spaced6_4_ridge':
  path=CACHE/f'{cell}_spaced_train_val.npz';meta=CACHE/f'{cell}_spaced_features_meta.json'
  if path.exists() and meta.exists():
   assert json.loads(meta.read_text())=={'lock':LOCK,'sha256':fsha(path)}
   all_x=sparse.load_npz(path)
   assert all_x.shape==(len(tr)+len(va),3840)
  else:
   all_x=sparse.vstack([feature_spaced(tr.sequence),feature_spaced(va.sequence)],format='csr')
   tmp=path.with_suffix('.part.npz');sparse.save_npz(tmp,all_x);os.replace(tmp,path)
   write_atomic_json({'lock':LOCK,'sha256':fsha(path)},meta)
  x=all_x[:len(tr)];v=all_x[len(tr):]
  normalizer=TfidfTransformer(norm='l2').fit(x)
  x=normalizer.transform(x);v=normalizer.transform(v)
  grid=CONFIG['classical_alphas']['spaced6_4']
 else:raise ValueError(arm)
 best=None
 for alpha in grid:
  model=Ridge(alpha=alpha,solver='lsqr' if sparse.issparse(x) else 'auto').fit(x,y)
  pred=model.predict(v);r=correlation(va.mean_log2_activity,pred)
  print(cell,arm,'alpha',alpha,'validation Pearson',round(r,4),flush=True)
  if best is None or r>best[0]:best=(r,alpha,pred)
 choice={'cell_line':cell,'model':arm,'selected_alpha':best[1],
         'validation_pearson':best[0],'selected_epoch':None}
 save_validation(cell,arm,va,best[2],choice)
 del x,v,normalizer
 gc.collect()
 return choice
BASE_CHOICES=[classical(cell,arm) for cell in ('HepG2','K562') for arm in BASE_ARMS]


REUSED baseline selection HepG2 gc_cpg_ridge
REUSED baseline selection HepG2 spaced6_4_ridge
REUSED baseline selection K562 gc_cpg_ridge
REUSED baseline selection K562 spaced6_4_ridge


In [4]:
# 3 — CNN: independent three-seed, train-only scaling; validation epoch selection.
import torch
from torch import nn
from torch.utils.data import Dataset,DataLoader
assert torch.cuda.is_available(),'Select a Kaggle GPU accelerator.'
DEVICE=torch.device('cuda:0')
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
try:torch.use_deterministic_algorithms(True,warn_only=True)
except Exception:pass

def set_seed(seed):
 random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
class CNN(nn.Module):
 def __init__(self):
  super().__init__()
  self.conv=nn.Sequential(nn.Conv1d(4,64,15,padding=7),nn.GELU(),
   nn.Conv1d(64,96,7,padding=3),nn.GELU(),nn.Conv1d(96,96,7,padding=3),nn.GELU())
  self.dropout=nn.Dropout(.2);self.head=nn.Sequential(nn.Linear(192,64),nn.GELU(),nn.Linear(64,1))
 def forward(self,x):
  h=self.conv(x);pooled=torch.cat([h.mean(-1),h.amax(-1)],dim=1)
  return self.head(self.dropout(pooled)).squeeze(-1)
def onehot(seqs):
 codes=np.stack([BASE_MAP[np.frombuffer(s.encode(),dtype=np.uint8)] for s in seqs])
 eye=np.eye(4,dtype=np.float32)
 return torch.from_numpy(eye[codes].transpose(0,2,1).copy()).to(DEVICE)
class OrientedRows(Dataset):
 def __init__(self,frame,mean,scale,seed):
  self.seqs=frame.sequence.tolist();self.y=((frame.mean_log2_activity.to_numpy(dtype=np.float32)-mean)/scale)
  self.seed=seed;self.epoch=0
 def __len__(self):return len(self.seqs)
 def __getitem__(self,i):
  seq=self.seqs[i]
  if (self.seed+17*self.epoch+15485863*i)%2:seq=rc(seq)
  return seq,float(self.y[i])
def cnn_predict(model,seqs,mean,scale):
 model.eval();pred=[]
 with torch.inference_mode():
  for i in range(0,len(seqs),CONFIG['cnn_batch']):
   batch=seqs[i:i+CONFIG['cnn_batch']]
   p=model(onehot(batch)).float();q=model(onehot([rc(s) for s in batch])).float()
   pred.extend((((p+q)/2)*scale+mean).cpu().numpy())
 return np.asarray(pred,dtype=float)
def cnn_seed(cell,seed):
 arm=f'cnn_seed_{seed}';tr,va,te=partitions(cell)
 ckpt=WEIGHTS/f'{cell}_{arm}_best.pt';cached=load_validation(cell,arm,va,ckpt)
 if cached:
  print('REUSED CNN',cell,seed);return cached[1]
 set_seed(seed);mean=float(tr.mean_log2_activity.mean());scale=float(tr.mean_log2_activity.std(ddof=0))
 assert scale>0
 net=CNN().to(DEVICE);rows=OrientedRows(tr,mean,scale,seed)
 loader=DataLoader(rows,batch_size=CONFIG['cnn_batch'],shuffle=True,num_workers=0,
  collate_fn=lambda items:(onehot([s for s,_ in items]),torch.tensor([y for _,y in items],device=DEVICE,dtype=torch.float32)),
  generator=torch.Generator().manual_seed(seed))
 opt=torch.optim.AdamW(net.parameters(),lr=CONFIG['cnn_lr'],weight_decay=.01)
 scaler=torch.amp.GradScaler('cuda');best=(-float('inf'),None,None)
 for epoch in range(1,CONFIG['cnn_epochs']+1):
  rows.epoch=epoch;net.train();losses=[]
  for xb,yb in loader:
   opt.zero_grad(set_to_none=True)
   with torch.autocast('cuda',dtype=torch.float16):loss=nn.functional.mse_loss(net(xb).float(),yb)
   assert torch.isfinite(loss)
   scaler.scale(loss).backward();scaler.unscale_(opt)
   nn.utils.clip_grad_norm_(net.parameters(),1.)
   scaler.step(opt);scaler.update();losses.append(float(loss.detach().cpu()))
  pv=cnn_predict(net,va.sequence.tolist(),mean,scale)
  r=correlation(va.mean_log2_activity,pv)
  print(cell,arm,'epoch',epoch,'loss',round(float(np.mean(losses)),4),'validation Pearson',round(r,4),flush=True)
  if r>best[0]:
   best=(r,epoch,pv)
   tmp=ckpt.with_suffix('.part')
   torch.save({'lock':LOCK,'cell_line':cell,'seed':seed,'epoch':epoch,
    'target_mean':mean,'target_scale':scale,'architecture':'cnn_v1',
    'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()}},tmp)
   os.replace(tmp,ckpt)
 assert best[1] is not None
 choice={'cell_line':cell,'model':arm,'selected_alpha':None,
         'selected_epoch':best[1],'validation_pearson':best[0]}
 save_validation(cell,arm,va,best[2],choice,ckpt)
 del net,opt,loader,rows,scaler;gc.collect();torch.cuda.empty_cache()
 return choice
CNN_CHOICES=[cnn_seed(cell,seed) for cell in ('HepG2','K562') for seed in SEEDS]


REUSED CNN HepG2 20260930
REUSED CNN HepG2 20260931
REUSED CNN HepG2 20260932
REUSED CNN K562 20260930
REUSED CNN K562 20260931
REUSED CNN K562 20260932


In [5]:
# 4 — Pinned DNABERT-2 regression: independent three-seed fine-tunes.
from importlib.metadata import version,PackageNotFoundError
try:installed=version('transformers')
except PackageNotFoundError:installed='missing'
if installed!='4.44.2':
 assert 'transformers' not in sys.modules,'Restart Kaggle session: Transformers imported before pinning.'
 subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.44.2'],check=True,timeout=900)
from transformers import AutoTokenizer,AutoModel
import transformers
assert transformers.__version__=='4.44.2'
def backbone():
 remote='zhihan1996/DNABERT-2-117M'
 tok=AutoTokenizer.from_pretrained(remote,revision=REVISION,trust_remote_code=True)
 try:enc=AutoModel.from_pretrained(remote,revision=REVISION,trust_remote_code=True)
 except (ValueError,TypeError):
  from transformers.models.bert.configuration_bert import BertConfig
  cfg=BertConfig.from_pretrained(remote,revision=REVISION)
  enc=AutoModel.from_pretrained(remote,revision=REVISION,trust_remote_code=True,config=cfg)
 import importlib
 mod=importlib.import_module(type(enc).__module__)
 assert hasattr(mod,'BertUnpadSelfAttention') and hasattr(mod,'flash_attn_qkvpacked_func')
 mod.flash_attn_qkvpacked_func=None
 return tok,enc
class Regressor(nn.Module):
 def __init__(self,enc):
  super().__init__();self.encoder=enc
  self.dropout=nn.Dropout(.1);self.head=nn.Linear(int(enc.config.hidden_size),1)
 def forward(self,b):
  output=self.encoder(**b);h=getattr(output,'last_hidden_state',None)
  if h is None:h=output[0]
  mask=b['attention_mask'].to(h.dtype).unsqueeze(-1)
  pooled=(h*mask).sum(1)/mask.sum(1).clamp(min=1)
  return self.head(self.dropout(pooled)).squeeze(-1)
def tokenize(tok,seqs):
 b=tok(list(seqs),padding=True,add_special_tokens=True,truncation=False,return_tensors='pt')
 assert b['input_ids'].shape[1]<=CONFIG['dnabert2_max_tokens'],b['input_ids'].shape
 return {k:v.to(DEVICE) for k,v in b.items() if k in ('input_ids','attention_mask')}
def bert_predict(net,tok,seqs,mean,scale):
 net.eval();out=[]
 with torch.inference_mode():
  for i in range(0,len(seqs),24):
   batch=seqs[i:i+24]
   x=tokenize(tok,batch);xr=tokenize(tok,[rc(s) for s in batch])
   with torch.autocast('cuda',dtype=torch.float16):
    p=net(x).float();q=net(xr).float()
   out.extend(((((p+q)/2)*scale)+mean).cpu().numpy())
 return np.asarray(out,dtype=float)
def bert_seed(cell,seed):
 arm=f'dnabert2_seed_{seed}';tr,va,te=partitions(cell)
 ckpt=WEIGHTS/f'{cell}_{arm}_best.pt';cached=load_validation(cell,arm,va,ckpt)
 if cached:
  print('REUSED DNABERT-2',cell,seed);return cached[1]
 set_seed(seed);mean=float(tr.mean_log2_activity.mean());scale=float(tr.mean_log2_activity.std(ddof=0))
 assert scale>0
 tok,enc=backbone();net=Regressor(enc).to(DEVICE)
 rows=OrientedRows(tr,mean,scale,seed)
 def collate(items):return tokenize(tok,[s for s,_ in items]),torch.tensor([y for _,y in items],device=DEVICE,dtype=torch.float32)
 loader=DataLoader(rows,batch_size=CONFIG['dnabert2_batch'],shuffle=True,num_workers=0,
  collate_fn=collate,generator=torch.Generator().manual_seed(seed))
 opt=torch.optim.AdamW(net.parameters(),lr=CONFIG['dnabert2_lr'],weight_decay=.01)
 scaler=torch.amp.GradScaler('cuda');best=(-float('inf'),None,None)
 for epoch in range(1,CONFIG['dnabert2_epochs']+1):
  rows.epoch=epoch;net.train();opt.zero_grad(set_to_none=True);losses=[]
  for step,(xb,yb) in enumerate(loader):
   with torch.autocast('cuda',dtype=torch.float16):loss=nn.functional.mse_loss(net(xb).float(),yb)
   assert torch.isfinite(loss),(cell,seed,epoch,step)
   scaler.scale(loss/CONFIG['dnabert2_accumulation']).backward()
   if (step+1)%CONFIG['dnabert2_accumulation']==0 or step+1==len(loader):
    scaler.unscale_(opt);nn.utils.clip_grad_norm_(net.parameters(),1.)
    scaler.step(opt);scaler.update();opt.zero_grad(set_to_none=True)
   losses.append(float(loss.detach().cpu()))
  pv=bert_predict(net,tok,va.sequence.tolist(),mean,scale)
  r=correlation(va.mean_log2_activity,pv)
  print(cell,arm,'epoch',epoch,'loss',round(float(np.mean(losses)),4),'validation Pearson',round(r,4),flush=True)
  if r>best[0]:
   best=(r,epoch,pv)
   tmp=ckpt.with_suffix('.part')
   torch.save({'lock':LOCK,'revision':REVISION,'cell_line':cell,'seed':seed,'epoch':epoch,
    'target_mean':mean,'target_scale':scale,
    'state_dict':{k:v.detach().cpu() for k,v in net.state_dict().items()}},tmp)
   os.replace(tmp,ckpt)
 assert best[1] is not None
 choice={'cell_line':cell,'model':arm,'selected_alpha':None,
         'selected_epoch':best[1],'validation_pearson':best[0]}
 save_validation(cell,arm,va,best[2],choice,ckpt)
 del net,enc,tok,opt,loader,rows,scaler;gc.collect();torch.cuda.empty_cache()
 return choice
BERT_CHOICES=[bert_seed(cell,seed) for cell in ('HepG2','K562') for seed in SEEDS]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 85.2 MB/s eta 0:00:00
REUSED DNABERT-2 HepG2 20260930
REUSED DNABERT-2 HepG2 20260931
REUSED DNABERT-2 HepG2 20260932
REUSED DNABERT-2 K562 20260930
REUSED DNABERT-2 K562 20260931
REUSED DNABERT-2 K562 20260932


In [6]:
# 5 — Final locked test inference after ALL validation choices are complete.
# Reuse any finished test prediction and authenticate its record order.
for cell in ('HepG2','K562'):
 tr,va,te=partitions(cell)
 for arm in BASE_ARMS+NEURAL_ARMS:
  assert load_validation(cell,arm,va,WEIGHTS/f'{cell}_{arm}_best.pt' if arm in NEURAL_ARMS else None)

def save_test(cell,arm,te,pred):
 assert len(pred)==len(te) and np.isfinite(pred).all()
 p=PRED/f'{cell}_{arm}_test.csv';m=CACHE/f'{cell}_{arm}_test_meta.json'
 write_atomic_csv(pd.DataFrame({'record_id':te.record_id,'prediction':pred}),p)
 write_atomic_json({'lock':LOCK,'sha256':fsha(p),'model':arm,'cell_line':cell},m)
def load_test(cell,arm,te):
 p=PRED/f'{cell}_{arm}_test.csv';m=CACHE/f'{cell}_{arm}_test_meta.json'
 if not(p.exists() and m.exists()):return None
 info=json.loads(m.read_text());assert info['lock']==LOCK and info['sha256']==fsha(p)
 x=pd.read_csv(p,dtype={'record_id':str,'prediction':float})
 assert x.record_id.tolist()==te.record_id.tolist() and np.isfinite(x.prediction).all()
 return x.prediction.to_numpy()
def baseline_test(cell,arm):
 tr,va,te=partitions(cell)
 if load_test(cell,arm,te) is not None:return
 choice=load_validation(cell,arm,va)[1]
 if arm=='gc_cpg_ridge':
  x=feature_gc(tr.sequence);t=feature_gc(te.sequence)
  normalizer=StandardScaler().fit(x);x=normalizer.transform(x);t=normalizer.transform(t)
 else:
  all_x=sparse.load_npz(CACHE/f'{cell}_spaced_train_val.npz');x=all_x[:len(tr)]
  t=feature_spaced(te.sequence)
  normalizer=TfidfTransformer(norm='l2').fit(x);x=normalizer.transform(x);t=normalizer.transform(t)
 model=Ridge(alpha=choice['selected_alpha'],solver='lsqr' if sparse.issparse(x) else 'auto').fit(x,tr.mean_log2_activity)
 save_test(cell,arm,te,model.predict(t));del x,t,model,normalizer;gc.collect()
for cell in ('HepG2','K562'):
 for arm in BASE_ARMS:baseline_test(cell,arm)
 for seed in SEEDS:
  arm=f'cnn_seed_{seed}';tr,va,te=partitions(cell)
  if load_test(cell,arm,te) is None:
   ckpt=torch.load(WEIGHTS/f'{cell}_{arm}_best.pt',map_location='cpu',weights_only=True)
   assert ckpt['lock']==LOCK and ckpt['cell_line']==cell and ckpt['seed']==seed
   net=CNN().to(DEVICE);net.load_state_dict(ckpt['state_dict'],strict=True)
   save_test(cell,arm,te,cnn_predict(net,te.sequence.tolist(),ckpt['target_mean'],ckpt['target_scale']))
   del net,ckpt;torch.cuda.empty_cache();gc.collect()
  arm=f'dnabert2_seed_{seed}'
  if load_test(cell,arm,te) is None:
   ckpt=torch.load(WEIGHTS/f'{cell}_{arm}_best.pt',map_location='cpu',weights_only=True)
   assert ckpt['lock']==LOCK and ckpt['revision']==REVISION and ckpt['cell_line']==cell and ckpt['seed']==seed
   tok,enc=backbone();net=Regressor(enc).to(DEVICE);net.load_state_dict(ckpt['state_dict'],strict=True)
   save_test(cell,arm,te,bert_predict(net,tok,te.sequence.tolist(),ckpt['target_mean'],ckpt['target_scale']))
   del tok,enc,net,ckpt;torch.cuda.empty_cache();gc.collect()
 print('All locked test scores ready for',cell,flush=True)


All locked test scores ready for HepG2
All locked test scores ready for K562


In [7]:
# 6 — Record-level predictions, metrics and paired locus bootstrap.
rows=[];choices=[]
for cell in ('HepG2','K562'):
 tr,va,te=partitions(cell)
 for arm in BASE_ARMS+NEURAL_ARMS:
  val,choice=load_validation(cell,arm,va,WEIGHTS/f'{cell}_{arm}_best.pt' if arm in NEURAL_ARMS else None)
  test=load_test(cell,arm,te);assert test is not None
  choices.append(choice)
  for frame,pred,split in ((va,val,'validation'),(te,test,'test')):
   rows.extend({'record_id':rid,'cell_line':cell,'chrom':chrom,'locus_group_id':group,
    'split':'test' if split=='test' else 'validation','model':arm,'target':float(y),'prediction':float(p)}
    for rid,chrom,group,y,p in zip(frame.record_id,frame.chrom,frame.locus_group_id,frame.mean_log2_activity,pred))
 for family in ('cnn','dnabert2'):
  for frame,split in ((va,'validation'),(te,'test')):
   vectors=[load_validation(cell,f'{family}_seed_{seed}',va,WEIGHTS/f'{cell}_{family}_seed_{seed}_best.pt')[0]
    if split=='validation' else load_test(cell,f'{family}_seed_{seed}',te) for seed in SEEDS]
   ensemble=np.stack(vectors).mean(axis=0)
   rows.extend({'record_id':rid,'cell_line':cell,'chrom':chrom,'locus_group_id':group,
    'split':split,'model':f'{family}_three_seed_mean','target':float(y),'prediction':float(p)}
    for rid,chrom,group,y,p in zip(frame.record_id,frame.chrom,frame.locus_group_id,frame.mean_log2_activity,ensemble))
predictions=pd.DataFrame(rows)
assert not predictions.duplicated(['record_id','model']).any()
assert set(predictions.model)==set(ALL_ARMS)
for cell in ('HepG2','K562'):
 for arm in ALL_ARMS:
  part=predictions[(predictions.cell_line==cell)&(predictions.model==arm)]
  assert len(part)==sum(len(x) for x in partitions(cell)[1:])
metrics=pd.DataFrame([{'cell_line':cell,'split':split,'model':arm,**measure(part.target,part.prediction)}
 for (cell,split,arm),part in predictions.groupby(['cell_line','split','model'])])
print(metrics[metrics.split.eq('test')].sort_values(['cell_line','pearson'],ascending=[True,False])[
 ['cell_line','model','n','pearson','spearman','r2','rmse','mae']].to_string(index=False))
# Paired, within-cell chromosome-test bootstrap over locus groups, no model refitting.
boot=[];rng=np.random.default_rng(CONFIG['bootstrap_seed'])
for cell in ('HepG2','K562'):
 sub=predictions[(predictions.cell_line==cell)&(predictions.split=='test')]
 wide=sub.pivot(index='record_id',columns='model',values='prediction')
 truth=sub[sub.model=='spaced6_4_ridge'].set_index('record_id').loc[wide.index]
 loci=truth.locus_group_id.to_numpy();groups=pd.Series(np.arange(len(loci))).groupby(loci).apply(lambda v:v.to_numpy()).tolist()
 y=truth.target.to_numpy();b=wide['spaced6_4_ridge'].to_numpy();m=wide['dnabert2_three_seed_mean'].to_numpy()
 point=correlation(y,m)-correlation(y,b)
 differences=[]
 for rep in range(CONFIG['bootstrap_repeats']):
  selected=rng.integers(0,len(groups),size=len(groups))
  index=np.concatenate([groups[i] for i in selected]);differences.append(correlation(y[index],m[index])-correlation(y[index],b[index]))
 boot.append({'cell_line':cell,'contrast':'dnabert2_three_seed_mean minus spaced6_4_ridge',
  'pearson_difference':point,'bootstrap_repeats':len(differences),
  'lower_95':float(np.quantile(differences,.025)),'upper_95':float(np.quantile(differences,.975)),
  'resampling_unit':'test locus_group_id'})
bootstrap=pd.DataFrame(boot);print(bootstrap.to_string(index=False))


cell_line                    model    n  pearson  spearman        r2     rmse      mae
    HepG2 dnabert2_three_seed_mean 6725 0.601991  0.598137  0.341641 0.572187 0.436989
    HepG2   dnabert2_seed_20260931 6725 0.598315  0.595739  0.356392 0.565741 0.424296
    HepG2   dnabert2_seed_20260932 6725 0.594726  0.596174  0.340364 0.572742 0.435566
    HepG2   dnabert2_seed_20260930 6725 0.582121  0.577073  0.258434 0.607270 0.479158
    HepG2          spaced6_4_ridge 6725 0.526938  0.534239  0.277601 0.599371 0.454370
    HepG2        cnn_seed_20260932 6725 0.522482  0.526764  0.247884 0.611574 0.453063
    HepG2      cnn_three_seed_mean 6725 0.509531  0.514945  0.252240 0.609801 0.458113
    HepG2        cnn_seed_20260930 6725 0.491859  0.498397  0.240244 0.614672 0.463576
    HepG2        cnn_seed_20260931 6725 0.489794  0.492748  0.236960 0.616000 0.467789
    HepG2             gc_cpg_ridge 6725 0.145590  0.168301  0.021169 0.697687 0.538828
     K562 dnabert2_three_seed_mean 8661 0.6

In [8]:
# 7 — Package auditable results AND all model weights. Keep both ZIPs.
ART=WORK/'results';ART.mkdir(exist_ok=True)
files={}
def save_csv(name,frame):
 p=ART/name;write_atomic_csv(frame,p);files[name]=p
save_csv('phase9c_predictions_validation_test.csv',predictions)
save_csv('phase9c_metrics.csv',metrics)
save_csv('phase9c_validation_choices.csv',pd.DataFrame(choices))
save_csv('phase9c_locus_bootstrap.csv',bootstrap)
save_csv('phase9c_split_summary.csv',d.groupby(['cell_line','split_9a'],as_index=False).agg(
 rows=('record_id','size'),mean_activity=('mean_log2_activity','mean'),std_activity=('mean_log2_activity','std')))
checkpoints={f'{cell}_{family}_seed_{seed}_best.pt':WEIGHTS/f'{cell}_{family}_seed_{seed}_best.pt'
 for cell in ('HepG2','K562') for family in ('cnn','dnabert2') for seed in SEEDS}
assert len(checkpoints)==12 and all(p.exists() for p in checkpoints.values())
report={'protocol_sha256':LOCK,'config':CONFIG,'phase9a_audit_sha256':sha(raw_audit),
 'phase9a_data_sha256':sha(raw_data),'phase9a_split_sha256':sha(raw_split),
 'validation_only_selection':'Ridge alpha or neural epoch by validation Pearson; three seeds declared upfront',
 'test_evaluated':'after all validation selections completed; no test-driven refit',
 'result_sha256':{name:fsha(p) for name,p in files.items()},
 'checkpoints':{name:{'sha256':fsha(p),'bytes':p.stat().st_size} for name,p in checkpoints.items()},
 'caveat':'HepG2 and K562 use different design families. Locus-group holdout does not rule out every near-duplicate or distant homolog.'}
rp=ART/'phase9c_provenance.json';rp.write_text(json.dumps(report,indent=2,sort_keys=True)+'\n');files[rp.name]=rp
OUT=Path('/kaggle/working/enhancer_phase9c_regression_results.zip')
FULL=Path('/kaggle/working/enhancer_phase9c_complete_with_weights.zip')
with zipfile.ZipFile(OUT,'w',allowZip64=True) as z:
 for name,p in files.items():z.write(p,name,compress_type=zipfile.ZIP_DEFLATED)
with zipfile.ZipFile(OUT) as z:
 assert z.testzip() is None
 for name,digest in report['result_sha256'].items():assert sha(z.read(name))==digest
with zipfile.ZipFile(FULL,'w',allowZip64=True) as z:
 z.write(OUT,OUT.name,compress_type=zipfile.ZIP_STORED)
 for name,p in checkpoints.items():z.write(p,name,compress_type=zipfile.ZIP_STORED)
 z.writestr('phase9c_complete_manifest.json',json.dumps({'protocol_sha256':LOCK,
  'results_zip_sha256':fsha(OUT),'checkpoints':report['checkpoints']},sort_keys=True,indent=2)+'\n')
with zipfile.ZipFile(FULL) as z:
 assert z.testzip() is None
 assert sha(z.read(OUT.name))==fsha(OUT)
print('READY:',OUT,'SHA256:',fsha(OUT),'bytes:',OUT.stat().st_size)
print('READY:',FULL,'SHA256:',fsha(FULL),'bytes:',FULL.stat().st_size)
from IPython.display import FileLink,display
os.chdir('/kaggle/working')
display(FileLink(OUT.name));display(FileLink(FULL.name))


READY: /kaggle/working/enhancer_phase9c_regression_results.zip SHA256: 3f3662a7330f8dd5c36bf6b94dbe26aa2a2f8c31113a4bf291bc12ca35531606 bytes: 7658218
READY: /kaggle/working/enhancer_phase9c_complete_with_weights.zip SHA256: c943a5f7695d87147f14447e1b4cad64a06ed1204a623a3eacf4db366c864dc9 bytes: 2820628889


/kaggle/working/enhancer_phase9c_regression_results.zip

/kaggle/working/enhancer_phase9c_complete_with_weights.zip